# IITM Reinforcement Learning Course Project — Final Notebook

**Course:** IITM Web M.Tech Program — Course ID 6002W  
**Project:** Industrial Inventory Control using Reinforcement Learning  
**Roll number:** DA25M579  
**Assigned parameter variant:** V030

This notebook is the final reproducibility and evidence record for the five distinct techniques used on the public leaderboard. It keeps the official environment and assigned configuration unchanged and uses official unshaped cost for reported validation.

## 1. Course requirements covered

The final notebook documents the assigned configuration, common state preprocessing and action mapping, technique-specific representations, important hyperparameters, reward shaping, training/local-validation methodology, convergence evidence, the five-technique comparison, final export/artifact mapping, and reproducibility checks.

The five distinct techniques are **PPO, DQN, Neural Network SARSA, A2C, and Double DQN**.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
TRAINING_ROOT = PROJECT_ROOT / "training_pipelines"
if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

CONFIG_PATH = TRAINING_ROOT / "assigned_config.json"
assigned_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

pd.Series({
    "roll_number": assigned_config["roll_number"],
    "variant_id": assigned_config["variant_id"],
    "config_fingerprint": assigned_config["config_fingerprint"],
})

## 2. Assigned configuration and official environment

The assigned V030 configuration is frozen in `training_pipelines/assigned_config.json`. The official observation contains current inventory, the arrival pipeline, demand history, day index, and capacity utilisation. The submission interface exposes only the documented observation and requires three order quantities in `{0, 10, ..., 100}`.

The official evaluator converts returned quantities to the environment's internal `MultiDiscrete([11, 11, 11])` action indices. No future demand, scenario identity, hidden parameter, `env.step()`, or `env.reset()` is available to `run_policy()`.

In [ ]:
from industrial_inventory_env import IndustrialInventoryEnv

env = IndustrialInventoryEnv(
    student_config=assigned_config,
    scenario_mode="random",
    domain_randomization=True,
)
obs, info = env.reset(seed=900)

print("Observation keys:", sorted(obs))
print("Inventory shape:", np.asarray(obs["inventory"]).shape)
print("Arrival pipeline shape:", np.asarray(obs["arrival_pipeline"]).shape)
print("Demand history shape:", np.asarray(obs["demand_history"]).shape)
print("Day:", obs["day"])
print("Capacity utilisation:", obs["capacity_utilisation"])
print("Action space:", env.action_space)
env.close()

## 3. Common representations and action mapping

Representation B is the 76-dimensional state used by PPO and Neural Network SARSA and by the original/final Double-DQN pipeline. It contains 38 normalized raw features plus 38 engineered features.

The final A2C policy uses Representation C (99 dimensions): Representation B plus EWMA demand, slope, coefficient of variation, recent minima/maxima, lead-time demand uncertainty, safety-stock signals, and weekly phase. All features are computed solely from the documented observation and early zero-padded demand is excluded from historical statistics.

Joint-action methods use base-11 encoding for the 1,331 possible combinations of per-product order indices.

In [ ]:
from training_pipelines.src.features.observation import RAW_FEATURE_DIM
from training_pipelines.src.features.engineered import REPRESENTATION_B_DIM
from training_pipelines.src.environment.action_codec import JOINT_ACTION_SIZE

print({
    "Representation A/raw dimension": RAW_FEATURE_DIM,
    "Representation B dimension": REPRESENTATION_B_DIM,
    "A2C Representation C dimension": 99,
    "joint action count": JOINT_ACTION_SIZE,
    "external quantities": list(range(0, 101, 10)),
})

## 4. Reward and validation

The official reward is negative daily total cost divided by 100. Where used, `ShapedReward` was applied only during training and annealed to zero over the training budget. Reported local and public performance is based on the official unshaped cost.

The official-style holdout is **40 seeds (900–939) × 5 scenarios = 200 episodes**, covering stationary, seasonal, trend, shock, and random cases.

## 5. Reproducible final training pipelines

The source tree retains the launchers and algorithm implementations needed to reproduce the final techniques. The commands below are the final training recipes; they are intentionally presented as reproducibility commands rather than executed automatically by this notebook.

In [ ]:
# PPO + Representation B
# python training_pipelines/training_scripts/train_ppo_rep_b_experiment.py --timesteps 500000 --learning-rate 0.0006 --entropy-coef 0.0 --seed 20260727 --device cpu --run-name final_ppo_rep_b_500k

# DQN
# python training_pipelines/training_scripts/train_dqn.py --timesteps 150000 --seed 20260727 --run-name final_dqn_150k

# Neural Network SARSA (retained public artifact)
# python training_pipelines/run_neural_sarsa_promote_v2.py

# A2C + Representation C
# python training_pipelines/training_scripts/train_a2c_rep_c.py --timesteps 500000 --n-envs 4 --seed 20260902 --learning-rate 0.0005 --gamma 0.99 --gae-lambda 0.95 --ent-coef 0.001 --n-steps 64 --vf-coef 0.5 --run-name final_a2c_rep_c_500k

# Double DQN
# python training_pipelines/training_scripts/train_double_dqn_experiment.py --timesteps 100000 --seed 20260727 --run-name final_double_dqn_100k

## 6. Final public leaderboard comparison

Lower cost is better. The A2C slot was replaced by the new Representation C candidate after an official 200-episode local mean of 94,180.86 and a public mean cost of 94,660.12. The final A2C training network is 256×256.

In [ ]:
public_results = pd.DataFrame({
    "Technique": ["PPO", "DQN", "A2C", "Neural Network SARSA", "Double DQN"],
    "Public mean cost": [83610.75, 90670.62, 94660.12, 113072.38, 115819.00],
    "Status": [
        "Retained",
        "Retained",
        "Final A2C replacement (Rep-C)",
        "Retained public artifact",
        "Retained public artifact",
    ],
})

public_results

In [ ]:
portfolio_average = float(public_results["Public mean cost"].mean())
print(f"Final public Top-5 average: {portfolio_average:,.2f}")

## 7. Final local validation evidence

The strongest preserved local holdout results are:

In [ ]:
local_results = pd.DataFrame({
    "Technique": ["PPO + Rep-B", "DQN", "A2C + Rep-C", "Neural Network SARSA"],
    "Official holdout mean cost": [83039.20, 96871.30, 94180.86, 111730.60],
    "Std": [9777.14, 11858.00, 7961.56, 9941.00],
    "Mean service": [0.992973, 0.988895, 0.994717, 0.992882],
})
local_results

### A2C Representation C official holdout

The final A2C candidate was evaluated on all 200 official holdout episodes. Scenario means were:

- Random: 94,911.88
- Seasonal: 92,426.00
- Shock: 95,611.69
- Stationary: 92,332.50
- Trend: 95,622.25

Overall: **94,180.86 mean cost**, **7,961.56 standard deviation**, **99.4717% mean service**.

### Final SARSA gate

The last targeted SARSA refinement (r2: learning rate 1.5e-4, γ=0.99, ε_end=0.01) achieved 124,306.75 during screening but only **117,164.21** on the official 200-episode holdout. It therefore did not replace the retained public SARSA artifact at 113,072.38.

### Double DQN gate

The targeted six-configuration Double DQN screen failed badly: the best screening configuration still had a cost above 1.2 million. The protected Double DQN artifact therefore remains unchanged at 115,819.00.

## 8. Learning/convergence evidence

Preserved learning-curve artifacts for PPO, DQN, and the actor-critic baseline remain available under `results/`. For Neural Network SARSA and the final A2C replacement, official 200-episode holdout summaries are the primary final validation evidence. No unsupported convergence curve is fabricated.

In [ ]:
# Example: display preserved baseline learning curves when the PNGs exist.
curve_paths = [
    PROJECT_ROOT / "results" / "ppo_learning_curve.png",
    PROJECT_ROOT / "results" / "dqn_learning_curve.png",
    PROJECT_ROOT / "results" / "a2c_learning_curve.png",
]
for path in curve_paths:
    if path.exists():
        print(path)
        display(plt.imread(path))

## 9. Final policy/artifact mapping

Each final submission contains a policy file and its colocated model artifact. The A2C `policy.py` uses the 99-D Representation C feature pipeline and loads `model.zip` from the same submission directory.

The public submission interface remains `run_policy(observation) -> [q1, q2, q3]`, with each quantity in `{0,10,...,100}`.

In [ ]:
artifact_map = pd.DataFrame({
    "Technique": ["PPO", "DQN", "Neural Network SARSA", "A2C", "Double DQN"],
    "Policy file": [
        "submissions/ppo/policy.py",
        "submissions/dqn/policy.py",
        "submissions/neural_sarsa/policy.py",
        "submissions/a2c/policy.py",
        "submissions/double_dqn/policy.py",
    ],
    "Model artifact": [
        "submissions/ppo/model.zip",
        "submissions/dqn/model.zip",
        "submissions/neural_sarsa/policy_state.pt",
        "submissions/a2c/model.zip",
        "submissions/double_dqn/model.zip",
    ],
})
artifact_map

## 10. Course-supplied policy validation

Run the supplied validator locally before portal upload. The validator checks importability, the exact `run_policy(observation)` interface, valid integer quantities, observation immutability, deterministic inference, absence of direct `env.step()`/`env.reset()` calls, and runtime limits.

In [ ]:
# Run in a terminal from the repository root:
#
# python policy_validation_tests.py submissions/ppo/policy.py
# python policy_validation_tests.py submissions/dqn/policy.py
# python policy_validation_tests.py submissions/neural_sarsa/policy.py
# python policy_validation_tests.py submissions/a2c/policy.py
# python policy_validation_tests.py submissions/double_dqn/policy.py

## 11. Final selection and reproducibility notes

The final portfolio is frozen as PPO, DQN, Neural Network SARSA, A2C + Representation C, and Double DQN. Failed exploratory algorithms and rejected tuning runs are not part of the final submission artifacts.

The official environment, assigned V030 configuration, official cost function, and leaderboard evaluator remain unchanged. The final code must be executed from the repository root so relative model paths resolve inside each submission package.